# 因子数据模型

因子数据模型采用了三层结构：
* 最上层是**因子库**：`QuantStudio.Factor.FactorDB.FactorDB`
* 因子库包含多张**因子表**：`QuantStudio.Factor.FactorTable.FactorTable`
* 每张因子表中又包含多个**因子**：`QuantStudio.Factor.Factor.Factor`

每张因子表的数据逻辑上是一个三维数组，第一维是因子名称，第二维是时间点，第三维是证券代码。具体到程序里的数据类型，以 `Panel`（`QuantStudio.Core.QSObject.Panel`）数据类型组织。QuantStudio 规定了这三个维度的先后顺序，对应于 Panel 数据类型，items 是因子，major_axis 是时点，minor_axis 是证券代码。在 QuantStudio 的所有 API 中，凡是涉及到因子数据的地方，都将遵守此组织原则。

每个因子的数据逻辑上是一个 DataFrame，index 是时间点，columns 是证券代码。

另外，对于时间点，采用 Python 的 datetime 表示；证券代码的数据类型为字符串，例如 "000001.SZ"；因子名称的数据类型也是字符串。

```mermaid
graph TB
    subgraph FactorFramework["因子框架"]
        direction TB

        FactorDB["因子库<br/>FactorDB"]

        FactorTable1["因子表1<br/>FactorTable"]
        FactorTable2["因子表2<br/>FactorTable"]

        subgraph FT1["因子表1内容"]
            F11["因子1<br/>Factor"]
            F12["因子2<br/>Factor"]
            F1M["因子M<br/>Factor"]
        end

        subgraph FT2["因子表2内容"]
            F21["因子1<br/>Factor"]
            F22["因子2<br/>Factor"]
            F2N["因子N<br/>Factor"]
        end
        
    end
    
    FactorDB --> FactorTable1
    FactorDB --> FactorTable2
    
    FactorTable1 --> F11
    FactorTable1 --> F12
    FactorTable1 --> F1M
    
    FactorTable2 --> F21
    FactorTable2 --> F22
    FactorTable2 --> F2N
    
    DF["DataFrame<br/>- index: 时间<br/>- columns: 证券代码"]
    
    F11 --> DF
    F12 --> DF
    F1M --> DF
    F21 --> DF
    F22 --> DF
    F2N --> DF

    style FactorFramework fill:#e1f5fe
    style FactorDB fill:#f87f89
    style FactorTable1 fill:#efbaf7
    style FactorTable2 fill:#efbaf7
    style DF fill:#fff3e0
```

# 因子与计算图

因子框架构建在计算图框架之上。每个因子（`Factor`）继承自计算图节点（`Node`），因此单个因子本质上是一个计算节点；多个因子之间的依赖关系构成有向无环计算图（DAG）。因子的数据获取最终由计算引擎驱动执行。

关于计算图的节点生命周期、Context、引擎调度等核心概念，请参见 **[计算图框架](../Core/计算图框架.ipynb)**；关于不同计算引擎的选择和并行策略，请参见 **[计算引擎](../Core/计算引擎.ipynb)**。

## 因子的两类角色

因子在计算图中扮演两种角色：

* **基础因子**：不依赖其他因子，数据直接来源于原始数据的因子。例如，通过 `FactorTable.getFactor()` 从因子表中获取的因子，以及通过 `DataFactor` 直接给入字面量数据的因子。基础因子是计算图的叶子节点。
* **衍生因子**：依赖其他因子（称为**描述子**），通过因子运算得到的因子。例如，市净率（PB）因子由总市值因子除以股东权益因子运算得到。衍生因子是计算图的内部节点，由算子（`FactorOperator`）作用于描述子产生。

## 因子上下文对象

因子框架在通用计算图上下文的基础上，定义了三个专用的上下文类型：

| 类型 | 基类 | 用途 |
|------|------|------|
| `FactorContext` | `Context` | 因子计算的全局上下文，新增 `DTRuler`（时点标尺）、`SectionIDs`（默认截面 ID）和 `DataCache`（因子数据缓存）字段 |
| `FactorLocalContext` | `DTLocalContext` | 因子计算的局部上下文，携带当前计算时点和 ID 序列信息 |
| `FactorInitData` | `DTInitData` | 因子计算的初始化数据，携带时点区间、截面 ID 和子因子名称信息 |

这三个类型定义在 `QuantStudio.Factor.Factor` 模块中，但在因子框架的各个组件（FactorTable、Factor、FactorOperator）中均被广泛使用。

# 因子运算

## 四类运算概述

因子的运算按照数据依赖范围划分为四类，每种运算的适用范围和效率各不相同：

| 运算类型 | 算子类 | 衍生因子类 | 依赖范围 | 典型应用 |
|----------|--------|------------|----------|----------|
| **单点运算** | `PointOperator` | `PointOperation` | 同时间点、同证券 | 估值指标（PB、PE 等） |
| **时序运算** | `TimeOperator` | `TimeOperation` | 历史时间序列、同证券 | 移动平均、EMA |
| **截面运算** | `SectionOperator` | `SectionOperation` | 同时间点、全截面证券 | 数据标准化（Z-score） |
| **面板运算** | `PanelOperator` | `PanelOperation` | 历史时间序列 + 全截面证券 | 时序截面双重标准化 |

每种因子的运算本质上是一个可调用对象（算子 `FactorOperator`），算子作用于描述子（依赖因子）后产生新的衍生因子。这些运算可以嵌套组合以形成更复杂的计算图。

详细的因子运算使用说明、参数配置和代码示例，请参见 **[因子开发](因子开发.ipynb)**。

## 运算符重载

`Factor` 类重载了常用 Python 运算符（`+`、`-`、`*`、`/`、`<`、`>`、`==`、`&`、`|`、`~` 等），使其可以直接作用于因子对象产生衍生因子。这些重载运算符定义在 `QuantStudio.Factor.BasicOperator` 中，本质上是基于单点运算的快捷方式。

`BasicOperator` 还提供了 `rename` 算子用于给因子重命名。

## 内置算子

`QuantStudio.Factor.FactorOperator` 模块预定义了一些常用的内置算子（如 `Log`、`Power` 等），可直接使用。

# 核心 API

## 因子库（FactorDB）

```python
class FactorDB(__QS_Object__):
    def connect(self) -> Self           # 连接到数据源
    def disconnect(self) -> int         # 断开连接
    def TableNames(self) -> List[str]   # 获取因子表名称列表
    def getTable(table_name, args)      # 获取因子表对象
```

`WritableFactorDB` 继承自 `FactorDB`，增加了写入和变更能力：`writeData`、`renameTable`、`deleteTable`、`renameFactor`、`deleteFactor`、`setTableMetaData`、`setFactorMetaData`。

## 因子表（FactorTable）

```python
class FactorTable(__QS_Object__):
    def FactorDB(self) -> FactorDB      # 所属因子库
    def FactorNames(self) -> List[str]  # 表中所有因子名称
    def getFactor(factor_name, args)    # 获取因子对象
    def getID(ifactor_name, idt)        # 获取 ID 序列
    def getDateTime(ifactor_name, ...)  # 获取时点序列
    def readData(factor_names, ids, dts)# 读取因子表数据，返回 Panel
    def getMetaData(key)                # 获取元信息
```

FactorTable 还实现了 `__getitem__`，支持 `ft["factor_name"]` 和 `ft[factor_names, dts, ids]` 两种索引方式。

## 因子（Factor）

```python
class Factor(Node):
    def FactorTable(self) -> FactorTable # 所属因子表
    def Descriptors(self) -> List[Factor]# 依赖的描述子列表
    def getID(idt)                       # 获取 ID 序列
    def getDateTime(iid, ...)            # 获取时点序列
    def readData(ids, dts)               # 读取因子数据，返回 DataFrame
    def getMetaData(key)                 # 获取元信息
    def new(args)                        # 创建同类型的新因子
```

`DataFactor(Factor)` 是直接赋予字面量数据（标量、DataFrame、Series）的因子，主要用于测试场景。它的 `readData` 直接返回构造时给定的数据。

# 可用因子库

QuantStudio 提供了多种因子库实现，用于连接不同的数据源：

| 因子库 | 模块 | 数据源 | 可写 | 文档 |
|--------|------|--------|------|------|
| `JYDB` | `QuantStudio.Factor.JYDB` | 聚源 PostgreSQL 数据库 | 否 | [JYDB](JYDB.ipynb) |
| `SQLDB` | `QuantStudio.Factor.SQLDB` | 通用 SQL 数据库 | 可选 | [SQLDB](SQLDB.ipynb) |
| `HDF5DB` | `QuantStudio.Factor.HDF5DB` | 本地 HDF5 文件 | 是 | [HDF5DB](HDF5DB.ipynb) |
| `BaoStockDB` | `QuantStudio.Factor.BaoStockDB` | BaoStock 在线 API | 否 | [BaoStockDB](BaoStockDB.ipynb) |

各因子库的具体连接方式、配置文件格式、支持的表类型和参数说明，请参见对应的文档。

# 因子存储

`FactorStorer`（`QuantStudio.Factor.FactorStorer`）是一个特殊的计算图节点，用于将计算后的衍生因子数据写入可写因子库（如 `HDF5DB`）中持久化保存。

```python
class FactorStorer(Node):
    # 参数：TargetFDB（目标因子库）、TargetTable（目标表名）、
    #       IfExists（写入方式：update/replace/append）
    # 依赖节点：待写入的因子列表
```

将 `FactorStorer` 作为依赖图的末端节点，与待存储的衍生因子一起交由引擎执行，即可完成因子数据的持久化。

# 相关文档索引

| 文档 | 内容 |
|------|------|
| [QuickStart](QuickStart.ipynb) | 因子框架快速入门示例 |
| [因子开发](因子开发.ipynb) | 因子运算详解（四类运算、算子定义、参数说明） |
| [JYDB](JYDB.ipynb) | 聚源数据库因子库的使用 |
| [HDF5DB](HDF5DB.ipynb) | 本地 HDF5 因子库的使用 |
| [SQLDB](SQLDB.ipynb) | 通用 SQL 因子库的使用 |
| [BaoStockDB](BaoStockDB.ipynb) | BaoStock 因子库的使用 |
| [计算图框架](../Core/计算图框架.ipynb) | 计算图节点、Context、生命周期 |
| [计算引擎](../Core/计算引擎.ipynb) | 引擎类型、并行策略 |
| [缓存](../Core/缓存.ipynb) | 因子缓存机制 |